# Activity Classification Using the UCI Daily and Sports Activities (DSA) Dataset

This notebook demonstrates how to use the DSA dataset within the PyHealth framework to train an LSTM classifier for two experimental setups:

1. **Binary classification** — one-vs-rest, replicating the paper's experimental protocol
2. **Multiclass classification** — all 19 activities simultaneously, extending the paper

**Dataset:** UCI Daily and Sports Activities  

**Paper:** Zhang et al. "Daily Physical Activity Monitoring — Adaptive Learning from Multi-source Motion Sensor Data". CHIL 2024 (PMLR 248:39-54) https://raw.githubusercontent.com/mlresearch/v248/main/assets/zhang24a/zhang24a.pdf

**Original Paper Code:** https://github.com/sunlabuiuc/PyHealth/blob/master/README.rst 

It may be noted that there are some difference between the implementation in the linked github and the actual paper. This replication attempts to follow the methodology outlined in the paper. Notably, paper suggest usage of the IPD metric to scale epochs, order the domains of transfer learning, and adaptively adjust the lr; however, the code provided does not seem to do these things as described.

**Model:** PyHealth built-in `RNN` (LSTM backbone), matching the architecture used in the paper's code (64 hidden units, dropout=0.2)

# Paper Replication

In [ ]:
import math
import random, time
import os
import contextlib
from typing import Dict, List, Tuple
from collections import defaultdict

import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader

from pyhealth.datasets.dsa import DSADataset
from pyhealth.datasets import collate_fn_dict_with_padding
from pyhealth.models import RNN
from pyhealth.tasks.dsa import (
    DSAActivityClassification,
    DSABinaryActivityClassification,
    compute_all_ipd_weights,
)

import warnings
warnings.simplefilter("ignore", FutureWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Configuration (paper's protocol) ────────────────────────────────
ALL_DOMAINS          = ['T', 'RA', 'LA', 'RL', 'LL']
TARGET_DOMAIN        = 'LA'
SOURCE_DOMAINS       = [d for d in ALL_DOMAINS if d != TARGET_DOMAIN]
METRIC               = 'dtw_classic'
KDE_BANDWIDTH        = 7.8
LR                   = 0.005
BATCH_SIZE           = 16
SOURCE_EPOCHS_NAIVE  = 30
TARGET_EPOCHS        = 30
EPOCH_SCALE_FACTOR   = 7.8
N_REPEATS            = 3   # paper: 15; use 3 for quick debugging
N_TRAIN_SUBJ         = 6
BASE_SEED            = 42
DATA_ROOT            = './data/DSA'
RANDOM_ACTIVITY      = True 

print(f'Config: {N_REPEATS} repeats, random_activity={RANDOM_ACTIVITY}')

# ── Load data ───────────────────────────────────────────────────────
print('\nLoading dataset...')
dataset = DSADataset(root=DATA_ROOT, download=True,
                     target_domain=TARGET_DOMAIN, scale=True)

# Will be reloaded per repeat if RANDOM_ACTIVITY=True, otherwise load once
domain_samples = {}
if not RANDOM_ACTIVITY:
    for domain in ALL_DOMAINS:
        task = DSABinaryActivityClassification(
            positive_activity_id=12, target_domain=domain)
        domain_samples[domain] = dataset.set_task(task)
    print(f'Loaded: {len(domain_samples[TARGET_DOMAIN])} samples in {TARGET_DOMAIN}')
else:
    print('RANDOM_ACTIVITY=True — will reload per repeat')


Device: cuda
Config: 3 repeats, random_activity=True

Loading dataset...
Download complete. Extracting ...
Extraction complete.
Dataset structure verified.
Indexed 9,120 segment files → ./data/DSA\dsa-metadata-pyhealth.csv
Initializing DSA dataset from ./data/DSA (dev mode: False)
No cache_dir provided. Using default cache dir: C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7ea
RANDOM_ACTIVITY=True — will reload per repeat


## Helper Functions

- **`evaluate_rcc`** — computes Ratio of Correct Classifications (the paper's primary metric, equivalent to accuracy). It handles both the binary sigmoid head (`y_prob` shape `(N, 1)`) and the multiclass softmax head (`(N, C)`).
- **`upsample_positives`** / **`downsample_negatives`** — implement the paper's class-balancing strategy: the training set upsamples the minority positive class to match the negative count; the test set downsamples negatives for balanced evaluation.
- **`build_aligned_domain_arrays`** — constructs aligned `(N, T, 1)` arrays for IPD computation by joining samples from all domains on their shared `pair_id`. Only training subjects are included to prevent data leakage into the distance computation.
- **`compute_ipd_weights_for_split`** — runs the full IPD pipeline for one subject split. The `reverse` flag switches between the paper's logic (more epochs for *more* similar domains, i.e. smaller IPD) and the author's code logic (more epochs for *larger* IPD values).

In [18]:
def move_to_device(batch, device):
    return {k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()}


def build_model(ref_samples, device):
    return RNN(dataset=ref_samples, rnn_type='LSTM',
               hidden_dim=64, dropout=0.2).to(device)


def make_dataloader(samples, batch_size, shuffle=True):
    return DataLoader(samples, batch_size=batch_size, shuffle=shuffle,
                      collate_fn=collate_fn_dict_with_padding)


def split_by_subject(samples, train_ids, test_ids):
    tr = [s for s in samples if s['patient_id'] in train_ids]
    te = [s for s in samples if s['patient_id'] in test_ids]
    return tr, te


def upsample_positives(samples, rng):
    pos = [s for s in samples if s['label'] == 1]
    neg = [s for s in samples if s['label'] == 0]
    if not pos or not neg: return samples
    n = len(neg)
    up = pos * (n // len(pos)) + rng.sample(pos, n % len(pos))
    return up + neg


def downsample_negatives(samples, rng):
    pos = [s for s in samples if s['label'] == 1]
    neg = [s for s in samples if s['label'] == 0]
    if not pos: return samples
    return pos + rng.sample(neg, min(len(pos), len(neg)))


def get_all_subject_ids(samples):
    return sorted(set(int(s['patient_id'][1:]) for s in samples))


def evaluate_rcc(model, loader, device) -> float:
    """RCC = ratio of correct classifications (accuracy).
    
    Handles PyHealth's binary sigmoid head (y_prob shape (N, 1)).
    """
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for batch in loader:
            batch  = move_to_device(batch, device)
            out    = model(**batch)
            y_prob = out['y_prob']

            # Handle both sigmoid (N, 1) and softmax (N, C) heads
            if y_prob.shape[-1] == 1:
                preds = (y_prob.squeeze(-1) >= 0.5).long()
            else:
                preds = y_prob.argmax(dim=1)

            # Coerce label to (N,) long
            raw = batch['label']
            if isinstance(raw, torch.Tensor):
                labels = raw.view(-1).long().to(device)
            else:
                labels = torch.tensor(raw, dtype=torch.long, device=device).view(-1)

            n        = preds.shape[0]
            correct += (preds == labels[:n]).sum().item()
            total   += n
    return correct / max(total, 1)


def run_epoch(model, loader, optimizer, device):
    model.train()
    for batch in loader:
        batch = move_to_device(batch, device)
        raw = batch.get('label')
        if raw is not None:
            if isinstance(raw, torch.Tensor):
                batch['label'] = raw.view(-1).long().to(device)
            else:
                batch['label'] = torch.tensor(
                    raw, dtype=torch.long, device=device).view(-1)
        optimizer.zero_grad()
        out = model(**batch)
        out['loss'].backward()
        optimizer.step()


def train_on_domain(model, loader, n_epochs, lr, device, verbose=False, label=''):
    if n_epochs <= 0: return
    opt = Adam(model.parameters(), lr=lr)
    for _ in range(n_epochs):
        run_epoch(model, loader, opt, device)
    if verbose and label:
        print(f'      {label}: {n_epochs} ep')


In [ ]:
def build_aligned_domain_arrays(
    domain_samples_dict: Dict[str, list],
    train_ids: set,
    channel: int = 0,
) -> Dict[str, np.ndarray]:
    """
    Extract aligned (N, T, 1) arrays for IPD computation.

    Each domain produces one array whose i-th row is the *same* physical
    activity instance as every other domain's i-th row (aligned by pair_id).
    Only training subjects are included.

    ``channel=0`` takes a single sensor axis, matching the author's code
    which squeezes each view to 1-D before computing distances.
    """
    # pair_id → time_series for each domain (training subjects only)
    domain_maps = {
        domain: {
            s['pair_id']: s['time_series']
            for s in samples if s['patient_id'] in train_ids
        }
        for domain, samples in domain_samples_dict.items()
    }

    # Intersection of pair_ids across all domains
    common = sorted(
        set.intersection(*[set(m.keys()) for m in domain_maps.values()])
    )
    if not common:
        raise ValueError('No common pair_ids found across domains.')

    arrays = {}
    for domain, pm in domain_maps.items():
        ts_list = [pm[pid][channel] for pid in common]   # each (T,)
        arr = np.array(ts_list, dtype=np.float32)         # (N, T)
        arrays[domain] = arr[:, :, np.newaxis]            # (N, T, 1)

    return arrays

def compute_ipd_weights_for_split(
    domain_samples_dict: Dict[str, list],
    train_ids: set,
    target_domain: str,
    metric: str = 'dtw_classic',
    bandwidth: float = 7.8,
    reverse: bool = True
) -> Tuple[Dict[str, float], Dict[str, int]]:
    """
    Full IPD pipeline for one subject split.

    Returns
    -------
    ipd_weights    : {domain: scalar weight}  (higher = less similar)
    weighted_epochs: {domain: epoch count for weighted pre-training}

    If reverse=True, allocate more epochs to *smaller* IPD values by using
    inverse weights, which is more consistent with the paper's written logic.
    If reverse=False, allocate more epochs to *larger* IPD values, matching
    the author's released code behavior.
    """
    print(f'    Building aligned domain arrays (metric={metric})...')
    domain_arrays = build_aligned_domain_arrays(
        domain_samples_dict, train_ids, channel=0
    )
    N = next(iter(domain_arrays.values())).shape[0]
    print(f'    Aligned {N} paired samples across {len(domain_arrays)} domains')

    print(f'    Computing pairwise distances (this may take a minute with DTW)...')
    ipd_weights = compute_all_ipd_weights(
        domain_data=domain_arrays,
        target_domain=target_domain,
        metric=metric,
        bandwidth=bandwidth,
    )

    # Epoch scaling: int(base * scale_factor * w / w_sum) + 1
    # Code from author uses "epochs=int(30*7*locals()[weight_name]/weight_all)+1"
    # This implies that higher weight → more epochs (matches *code*, opposite of paper §4.2)
    if reverse:
        eps = 1e-8
        epoch_weights = {
            d: 1.0 / max(w, eps)
            for d, w in ipd_weights.items()
        }
    else:
        epoch_weights = dict(ipd_weights)

    w_sum = sum(epoch_weights.values())
    weighted_epochs = (
        {
            d: int(SOURCE_EPOCHS_NAIVE * EPOCH_SCALE_FACTOR * epoch_weights[d] / w_sum) + 1
            for d in ipd_weights
        }
        if w_sum > 0
        else {d: SOURCE_EPOCHS_NAIVE for d in ipd_weights}
    )

    return ipd_weights, weighted_epochs

## Experiment Loop

**`run_one_repeat`** trains and evaluates all three conditions for a single subject split:

| Condition | Description |
|---|---|
| No Transfer | Train only on target domain (LA). Baseline — no cross-domain knowledge. |
| Naive Transfer | Pre-train on each source domain for a fixed epoch count, then fine-tune on target. |
| Weighted Transfer | Pre-train on sources ordered and weighted by IPD, then fine-tune on target. |

All three conditions use the same freshly initialised model to ensure comparability. The target domain training set is upsampled (positives repeated) and the test set is downsampled (negatives randomly removed) to produce balanced evaluation, following the paper's protocol.

**`run_experiment`** runs `run_one_repeat` for `N_REPEATS` independent random subject splits, accumulating results for mean and standard deviation reporting.

In [24]:
paper_weight = True

@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, "w") as fnull:
        with contextlib.redirect_stdout(fnull), contextlib.redirect_stderr(fnull):
            yield

def run_one_repeat(
    domain_samples:  Dict[str, list],
    train_ids:       set,
    test_ids:        set,
    ipd_weights:     Dict[str, float],
    weighted_epochs: Dict[str, int],
    source_domains:  List[str],
    target_domain:   str,
    device,
    rng:             random.Random,
) -> Dict[str, float]:
    loaders = {}
    for domain, samples in domain_samples.items():
        tr, te = split_by_subject(samples, train_ids, test_ids)
        if domain == target_domain:
            tr = upsample_positives(tr, rng)
            te = downsample_negatives(te, rng)
        loaders[domain] = (
            make_dataloader(tr, BATCH_SIZE, shuffle=True),
            make_dataloader(te, BATCH_SIZE, shuffle=False),
        )

    tgt_tr_ld, tgt_te_ld = loaders[target_domain]
    ref = domain_samples[target_domain]
    results = {}

    # No Transfer
    m = build_model(ref, device)
    train_on_domain(m, tgt_tr_ld, TARGET_EPOCHS, LR, device)
    results['No Transfer'] = evaluate_rcc(m, tgt_te_ld, device)

    # Naive Transfer
    m = build_model(ref, device)
    for src in source_domains:
        train_on_domain(m, loaders[src][0], SOURCE_EPOCHS_NAIVE, LR, device)
    train_on_domain(m, tgt_tr_ld, TARGET_EPOCHS, LR, device)
    results['Naive Transfer'] = evaluate_rcc(m, tgt_te_ld, device)

    # Weighted Transfer:
    # If paper_weight then sort descending (most similar first) and use epochs ∝ 1/IPD
    m = build_model(ref, device)
    sorted_sources = sorted(source_domains,
                            key=lambda d: ipd_weights.get(d, 0.),
                            reverse=paper_weight)
    for src in sorted_sources:
        n_ep = weighted_epochs.get(src, SOURCE_EPOCHS_NAIVE)
        train_on_domain(m, loaders[src][0], n_ep, LR, device)
    train_on_domain(m, tgt_tr_ld, TARGET_EPOCHS, LR, device)
    results['Weighted Transfer'] = evaluate_rcc(m, tgt_te_ld, device)

    return results


def run_experiment(
    n_repeats:      int = N_REPEATS,
    base_seed:      int = BASE_SEED,
    random_activity: bool = RANDOM_ACTIVITY,
) -> List[dict]:
    """Full experiment using paper's protocol."""
    all_subjects = None
    rng = random.Random(base_seed)
    all_activities = list(range(1, 20))
    all_results = []

    for rep in range(n_repeats):
        t0 = time.time()
        print(f'\n{"="*62}')
        print(f' Repeat {rep + 1} / {n_repeats}')
        print(f'{"="*62}')

        # Reload if random activity
        cur_domain_samples = {}
        if random_activity:
            act_id = rng.choice(all_activities)
            print(f' Positive activity: {act_id}')
            for domain in ALL_DOMAINS:
                task = DSABinaryActivityClassification(
                    positive_activity_id=act_id, target_domain=domain)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    with suppress_output():
                        cur_domain_samples[domain] = dataset.set_task(task)
        else:
            cur_domain_samples = domain_samples

        if all_subjects is None:
            all_subjects = get_all_subject_ids(cur_domain_samples[TARGET_DOMAIN])

        # Subject split
        train_subj = sorted(rng.sample(all_subjects, N_TRAIN_SUBJ))
        test_subj  = [s for s in all_subjects if s not in train_subj]
        train_ids  = {f'p{s}' for s in train_subj}
        test_ids   = {f'p{s}' for s in test_subj}
        print(f' Train: {train_subj}  Test: {test_subj}')

        # IPD
        print(' Computing IPD...')
        try:
            ipd_weights, w_epochs = compute_ipd_weights_for_split(
                cur_domain_samples, train_ids, TARGET_DOMAIN, METRIC, KDE_BANDWIDTH)
        except Exception as e:
            print(f'  IPD failed: {e}')
            ipd_weights = {d: 1. for d in SOURCE_DOMAINS}
            w_epochs = {d: SOURCE_EPOCHS_NAIVE for d in SOURCE_DOMAINS}

        desc_order = sorted(SOURCE_DOMAINS, key=lambda d: ipd_weights[d], reverse=True)
        print(f' IPD rank (desc): {" > ".join(f"{d}={ipd_weights[d]:.3f}" for d in desc_order)}')
        print(f' epochs: {w_epochs}')

        # Run experiment
        print(' Training...')
        rep_rng = random.Random(rng.random())
        rep_res = run_one_repeat(
            cur_domain_samples, train_ids, test_ids,
            ipd_weights, w_epochs,
            SOURCE_DOMAINS, TARGET_DOMAIN, device, rep_rng)

        rep_res.update({
            'repeat': rep + 1,
            'train_subjects': train_subj,
            'test_subjects': test_subj,
            'ipd_weights': dict(ipd_weights),
        })
        all_results.append(rep_res)

        elapsed = time.time() - t0
        print(f' Done in {elapsed:.0f}s')
        for cond in ('No Transfer', 'Naive Transfer', 'Weighted Transfer'):
            print(f'   {cond:<22}: {rep_res[cond]:.4f}')

    return all_results


In [21]:
results = run_experiment()


 Repeat 1 / 3
 Positive activity: 4
Setting task dsa_binary_classification for DSA base dataset...
Task cache paths: task_df=C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7ea\tasks\dsa_binary_classification_70e62abf-439f-527b-9f04-71b62644f742\task_df.ld, samples=C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7ea\tasks\dsa_binary_classification_70e62abf-439f-527b-9f04-71b62644f742\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Found cached processed samples at C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7ea\tasks\dsa_binary_classification_70e62abf-439f-527b-9f04-71b62644f742\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.
Setting task dsa_binary_classification for DSA base dataset...
Task cache paths: task_df=C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7ea\tasks\dsa_binary_classification_f68bfef6-b72d-

## Results

The table reports mean RCC and standard deviation across repeats. The paper (Table 1) reports 0.9722 ± 0.0104 for DTW-Paired with LSTM — our replication shows lower values for two reasons:

1. We use only 3 repeats rather than 15, so variance is higher.
2. The paper's reported results may reflect a fixed positive activity rather than a randomly selected one per repeat.

The **Weighted Transfer gain** row shows how much the IPD-weighted pre-training adds over the no-transfer baseline on average. A negative gain indicates that, for this small sample, the transfer procedure does not reliably help — consistent with the paper's RSS appendix results where transfer does not always improve over the no-transfer baseline.

In [22]:
CONDITIONS = ['No Transfer', 'Naive Transfer', 'Weighted Transfer']

print('\nPaper\'s Implementation — RCC by condition\n')
col_w = 18
header = f'{"Repeat":<8}' + ''.join(f'{c:>{col_w}}' for c in CONDITIONS)
print(header)
print('-' * (8 + col_w * len(CONDITIONS)))

for res in results:
    rep = res['repeat']
    row = f'Rep {rep:<4}'
    for cond in CONDITIONS:
        row += f'{res[cond]:>{col_w}.4f}'
    print(row)

print('-' * (8 + col_w * len(CONDITIONS)))
row = f'{"Mean":<8}'
for cond in CONDITIONS:
    vals = [r[cond] for r in results]
    row += f'{np.mean(vals):>{col_w}.4f}'
print(row)
row = f'{"Std":<8}'
for cond in CONDITIONS:
    vals = [r[cond] for r in results]
    row += f'{np.std(vals):>{col_w}.4f}'
print(row)

print(f'\nWeighted Transfer gain over No Transfer:')
wt_vals = [r['Weighted Transfer'] for r in results]
nt_vals = [r['No Transfer'] for r in results]
gain = np.mean(wt_vals) - np.mean(nt_vals)
print(f'  {gain:+.4f}  (std: {np.std([w - n for w, n in zip(wt_vals, nt_vals)]):.4f})')



Paper's Implementation — RCC by condition

Repeat         No Transfer    Naive Transfer Weighted Transfer
--------------------------------------------------------------
Rep 1               0.6333            0.6708            0.6125
Rep 2               0.7708            0.7875            0.8125
Rep 3               0.7333            0.6792            0.6833
--------------------------------------------------------------
Mean                0.7125            0.7125            0.7028
Std                 0.0580            0.0531            0.0828

Weighted Transfer gain over No Transfer:
  -0.0097  (std: 0.0382)


In my limited sample size I found that the weighted transfer did not perform better than no knowledge transfer. It should be noted that this example used only 3 repetitions (instead of the 15 of the paper) and the results are very high variance.

# Ablations

In [41]:
def evaluate_f1_binary(model, loader, device) -> float:
    from sklearn.metrics import f1_score
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            batch  = move_to_device(batch, device)
            out    = model(**batch)
            y_prob = out['y_prob']

            if y_prob.shape[-1] == 1:
                preds = (y_prob.squeeze(-1) >= 0.5).long()
            else:
                preds = y_prob.argmax(dim=1)

            raw = batch['label']
            if isinstance(raw, torch.Tensor):
                labels = raw.view(-1).long()
            else:
                labels = torch.tensor(raw, dtype=torch.long)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.tolist())

    return f1_score(all_labels, all_preds, average='macro', zero_division=0)


def majority_baseline_rcc(samples: list) -> float:
    """RCC achieved by always predicting the most common class."""
    from collections import Counter
    counts = Counter(s['label'] for s in samples)
    return counts.most_common(1)[0][1] / len(samples)


def train_one_pass(model, loader, optimizer, device):
    model.train()
    for batch in loader:
        batch = move_to_device(batch, device)
        optimizer.zero_grad()
        out = model(**batch)
        out['loss'].backward()
        optimizer.step()



In [42]:
def run_one_repeat_classification(
    domain_samples:  Dict[str, list],
    train_ids:       set,
    test_ids:        set,
    ipd_weights:     Dict[str, float],
    weighted_epochs: Dict[str, int],
    source_domains:  List[str],
    target_domain:   str,
    balance_train:   bool,
    balance_test:    bool,
    device,
    rng:             random.Random,
) -> Dict[str, Dict[str, float]]:
    """
    Run all three conditions for one subject split under a specific
    class-balance configuration.

    Returns
    -------
    dict  condition → {'rcc': float, 'f1': float}
    """
    loaders = {}
    test_sets = {}   # keep raw test samples for majority baseline

    for domain, samples in domain_samples.items():
        tr, te = split_by_subject(samples, train_ids, test_ids)

        if domain == target_domain:
            if balance_train:
                tr = upsample_positives(tr, rng)
            if balance_test:
                te = downsample_negatives(te, rng)
            test_sets[domain] = te   # save for baseline calc

        loaders[domain] = (
            make_dataloader(tr, BATCH_SIZE, shuffle=True),
            make_dataloader(te, BATCH_SIZE, shuffle=False),
        )

    tgt_tr_ld, tgt_te_ld = loaders[target_domain]
    ref = domain_samples[target_domain]
    results = {}

    for cond_name, train_fn in [
        ('No Transfer',       lambda m: None),
        ('Naive Transfer',    lambda m: [train_on_domain(m, loaders[s][0],
                                         SOURCE_EPOCHS_NAIVE, LR, device)
                                         for s in source_domains]),
        ('Weighted Transfer', lambda m: [train_on_domain(m, loaders[s][0],
                                         weighted_epochs.get(s, SOURCE_EPOCHS_NAIVE),
                                         LR, device)
                                         for s in sorted(source_domains,
                                             key=lambda d: ipd_weights.get(d, 0.),
                                             reverse=True)]),
    ]:
        m = build_model(ref, device)
        train_fn(m)                          # pre-train on source domains
        train_on_domain(m, tgt_tr_ld, TARGET_EPOCHS, LR, device)
        results[cond_name] = {
            'rcc': evaluate_rcc(m, tgt_te_ld, device),
            'f1':  evaluate_f1_binary(m, tgt_te_ld, device),
        }

    results['_majority_rcc'] = majority_baseline_rcc(test_sets[target_domain])
    return results


In [38]:
print('Loading 19-class samples for all domains...')
mc_domain_samples: Dict[str, list] = {}
for domain in ALL_DOMAINS:
    task = DSAActivityClassification(target_domain=domain)
    mc_domain_samples[domain] = dataset.set_task(task)
    print(f'  {domain}: {len(mc_domain_samples[domain])} samples, '
          f'{len(set(s["label"] for s in mc_domain_samples[domain]))} classes')


Loading 19-class samples for all domains...
Setting task dsa_activity_classification for DSA base dataset...
Task cache paths: task_df=C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7ea\tasks\dsa_activity_classification_df8f03e2-23d6-50b2-bd42-742d6eb0ec29\task_df.ld, samples=C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7ea\tasks\dsa_activity_classification_df8f03e2-23d6-50b2-bd42-742d6eb0ec29\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld
Found cached processed samples at C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7ea\tasks\dsa_activity_classification_df8f03e2-23d6-50b2-bd42-742d6eb0ec29\samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld, skipping processing.
  T: 9120 samples, 9120 classes
Setting task dsa_activity_classification for DSA base dataset...
Task cache paths: task_df=C:\Users\11400\AppData\Local\pyhealth\pyhealth\Cache\64d1ae8a-7617-555e-b299-2d3ab371f7

The four binary setups all frame the problem as one-vs-rest detection of a single activity, but differ in how they handle the 1:18 class imbalance:

Balanced (paper) — upsamples positives in training, downsamples negatives in testing. This is the paper's exact protocol and produces a fair 50/50 test set.

Unbalanced train — keeps the natural imbalance in training but still downsamples the test set. Tests whether the balanced test benefit survives an imbalanced training regime.

Natural test — natural imbalance in both train and test. The closest to real-world deployment conditions.

Fully unbalanced — natural imbalance everywhere with no rebalancing at any stage.

In [52]:
def build_aligned_domain_arrays(domain_samples_dict, train_ids, channel=0):
    domain_maps = {
        dom: {s['pair_id']: s['time_series']
              for s in samps if s['patient_id'] in train_ids}
        for dom, samps in domain_samples_dict.items()
    }
    common = sorted(set.intersection(*[set(m.keys()) for m in domain_maps.values()]))
    arrays = {}
    for dom, pm in domain_maps.items():
        arr = np.array([pm[pid][channel] for pid in common], dtype=np.float32)
        arrays[dom] = arr[:, :, np.newaxis]
    return arrays


def get_all_subject_ids(samples):
    return sorted(set(int(s['patient_id'][1:]) for s in samples))


# ── Classification ablation setups ───────────────────────────────────
# Each entry: (label, domain_samples_dict, balance_train, balance_test)
#
# 'Binary + natural test' deliberately omits test downsampling.
# RCC there is not comparable to other setups — majority-class baseline
# gives ~0.947 (18/19 samples are negative). Watch the F1 column instead.

CLASSIFICATION_SETUPS = [
    ('Binary + balanced (paper)',  mc_domain_samples,    True,  True),
    ('Binary + unbalanced train',  mc_domain_samples,    False, True),
    ('Binary + natural test',      mc_domain_samples,    True,  False),
    ('Binary + fully unbalanced',  mc_domain_samples,    False, False),
    # Multiclass handled separately below — different labels
]


def run_classification_ablation(
    n_repeats:   int = N_REPEATS,
    base_seed:   int = BASE_SEED,
) -> Dict[str, list]:
    """
    Run every classification setup for n_repeats independent subject splits.

    The subject splits and IPD weights are computed ONCE per repeat and
    then reused across all setups so results are directly comparable.

    Returns
    -------
    dict  setup_label → list of per-repeat result dicts
    """
    all_subjects = get_all_subject_ids(mc_domain_samples[TARGET_DOMAIN])
    rng = random.Random(base_seed)

    # One results list per setup
    results = {label: [] for label, *_ in CLASSIFICATION_SETUPS}
    results['Multiclass'] = []

    for rep in range(n_repeats):
        print(f'\n{"="*60}')
        print(f' Ablation A — Repeat {rep + 1} / {n_repeats}')
        print(f'{"="*60}')

        # ── Subject split (shared across all setups this repeat) ─────
        train_subj = sorted(rng.sample(all_subjects, N_TRAIN_SUBJ))
        test_subj  = [s for s in all_subjects if s not in train_subj]
        train_ids  = {f'p{s}' for s in train_subj}
        test_ids   = {f'p{s}' for s in test_subj}
        print(f' Train: {train_subj}  Test: {test_subj}')

        # ── IPD (shared; computed from binary samples, training subjs) ─
        print(' Computing IPD weights...')
        try:
            arrays = build_aligned_domain_arrays(mc_domain_samples, train_ids)
            ipd_weights = compute_all_ipd_weights(
                domain_data=arrays, target_domain=TARGET_DOMAIN,
                metric=METRIC, bandwidth=KDE_BANDWIDTH)
            w_sum = sum(ipd_weights.values())
            # divided the epochs by 3 to reduce computation time
            w_epochs = {
                d: math.ceil((int(SOURCE_EPOCHS_NAIVE * EPOCH_SCALE_FACTOR * w / w_sum) + 1) / 3)
                for d, w in ipd_weights.items()
            }
        except Exception as e:
            print(f'  IPD failed ({e}); using uniform weights.')
            ipd_weights = {d: 1. for d in SOURCE_DOMAINS}
            w_epochs    = {d: SOURCE_EPOCHS_NAIVE for d in SOURCE_DOMAINS}

        # ── Binary setups ────────────────────────────────────────────
        for label, samp_dict, bal_tr, bal_te in CLASSIFICATION_SETUPS:
            print(f' Running: {label}')
            rep_rng = random.Random(rng.random())  # deterministic but independent
            res = run_one_repeat_classification(
                domain_samples  = samp_dict,
                train_ids       = train_ids,
                test_ids        = test_ids,
                ipd_weights     = ipd_weights,
                weighted_epochs = w_epochs,
                source_domains  = SOURCE_DOMAINS,
                target_domain   = TARGET_DOMAIN,
                balance_train   = bal_tr,
                balance_test    = bal_te,
                device          = device,
                rng             = rep_rng,
            )
            results[label].append(res)
            for cond in ('No Transfer', 'Naive Transfer', 'Weighted Transfer'):
                r = res[cond]
                print(f'   {cond:<22}: RCC={r["rcc"]:.4f}  F1={r["f1"]:.4f}')

        # ── Multiclass ───────────────────────────────────────────────
        print(' Running: Multiclass (19-class)')
        mc_loaders = {}
        for domain, samps in mc_domain_samples.items():
            tr, te = split_by_subject(samps, train_ids, test_ids)
            mc_loaders[domain] = (
                make_dataloader(tr, BATCH_SIZE, shuffle=True),
                make_dataloader(te, BATCH_SIZE, shuffle=False),
            )
        mc_tgt_tr, mc_tgt_te = mc_loaders[TARGET_DOMAIN]
        mc_ref = mc_domain_samples[TARGET_DOMAIN]
        mc_res = {}

        for cond_name, src_list in [
            ('No Transfer',       []),
            ('Naive Transfer',    SOURCE_DOMAINS),
            ('Weighted Transfer', sorted(SOURCE_DOMAINS,
                                         key=lambda d: ipd_weights.get(d, 0.),
                                         reverse=True)),
        ]:
            m = build_model(mc_ref, device)
            for src in src_list:
                n_ep = w_epochs.get(src, SOURCE_EPOCHS_NAIVE)
                train_on_domain(m, mc_loaders[src][0], n_ep, LR, device)
            train_on_domain(m, mc_tgt_tr, TARGET_EPOCHS, LR, device)
            mc_res[cond_name] = {
                'rcc': evaluate_rcc(m, mc_tgt_te, device),
                'f1':  float('nan'),   # macro F1 for 19 classes not shown
            }
            print(f'   {cond_name:<22}: RCC={mc_res[cond_name]["rcc"]:.4f}')

        results['Multiclass'].append(mc_res)

    return results


In [54]:
classification_results = run_classification_ablation(n_repeats=1)


 Ablation A — Repeat 1 / 1
 Train: [1, 2, 3, 6, 7, 8]  Test: [4, 5]
 Computing IPD weights...
IPD weight T → LA: 5.9967 (metric=dtw_classic)
IPD weight RA → LA: 5.4693 (metric=dtw_classic)
IPD weight RL → LA: 7.2420 (metric=dtw_classic)
IPD weight LL → LA: 7.0441 (metric=dtw_classic)
 Running: Binary + balanced (paper)
   No Transfer           : RCC=0.5042  F1=0.5041
   Naive Transfer        : RCC=0.5750  F1=0.5731
   Weighted Transfer     : RCC=0.6000  F1=0.5982
 Running: Binary + unbalanced train
   No Transfer           : RCC=0.2542  F1=0.0555
   Naive Transfer        : RCC=0.0917  F1=0.0222
   Weighted Transfer     : RCC=0.1792  F1=0.0342
 Running: Binary + natural test
   No Transfer           : RCC=0.0513  F1=0.0104
   Naive Transfer        : RCC=0.1057  F1=0.0735
   Weighted Transfer     : RCC=0.0930  F1=0.0611
 Running: Binary + fully unbalanced
   No Transfer           : RCC=0.4377  F1=0.4300
   Naive Transfer        : RCC=0.4105  F1=0.3854
   Weighted Transfer     : RCC=0.42

## Results

In [57]:
CONDITIONS = ['No Transfer', 'Naive Transfer', 'Weighted Transfer']
ALL_SETUPS = [label for label, *_ in CLASSIFICATION_SETUPS] + ['Multiclass']

# ── RCC table ────────────────────────────────────────────────────────
print('RCC (Ratio of Correct Classifications)\n')
col_w = 24
header = f'{"Setup":<32}' + ''.join(f'{c:>{col_w}}' for c in CONDITIONS)
print(header)
print('-' * (32 + col_w * len(CONDITIONS)))

for setup in ALL_SETUPS:
    reps = classification_results[setup]
    row  = f'{setup:<32}'
    for cond in CONDITIONS:
        vals = [r[cond]['rcc'] for r in reps if cond in r]
        row += f'{np.mean(vals):.4f}'.rjust(col_w)
    print(row)

# ── F1 table (binary setups only) ────────────────────────────────────
print('\n\nMacro F1 (binary setups only; not distorted by class imbalance)\n')
header_f1 = f'{"Setup":<32}' + ''.join(f'{c:>{col_w}}' for c in CONDITIONS)
print(header_f1)
print('-' * (32 + col_w * len(CONDITIONS)))

for label, *_ in CLASSIFICATION_SETUPS:
    reps = classification_results[label]
    row  = f'{label:<32}'
    for cond in CONDITIONS:
        vals = [r[cond]['f1'] for r in reps if cond in r]
        row += f'{np.mean(vals):.4f}'.rjust(col_w)
    print(row)

# ── Majority-class baseline (for natural-test rows) ──────────────────
print('\n\nMajority-class baseline RCC (always predict most common class)\n')
for label, *_ in CLASSIFICATION_SETUPS:
    reps = classification_results[label]
    baselines = [r['_majority_rcc'] for r in reps]
    print(f'  {label:<40}: {np.mean(baselines):.4f}')

# ── Summary interpretation guide ─────────────────────────────────────
print('\n\nInterpretation guide:')
print('  Weighted-Transfer gain over No-Transfer in each setup:')
for setup in ALL_SETUPS:
    reps = classification_results[setup]
    nt_vals = [r['No Transfer']['rcc']       for r in reps if 'No Transfer' in r]
    wt_vals = [r['Weighted Transfer']['rcc'] for r in reps if 'Weighted Transfer' in r]
    gain = np.mean(wt_vals) - np.mean(nt_vals)
    print(f'  {setup:<40}: Δ = {gain:+.4f}')


RCC (Ratio of Correct Classifications)

Setup                                        No Transfer          Naive Transfer       Weighted Transfer
--------------------------------------------------------------------------------------------------------
Binary + balanced (paper)                         0.5042                  0.5750                  0.6000
Binary + unbalanced train                         0.2542                  0.0917                  0.1792
Binary + natural test                             0.0513                  0.1057                  0.0930
Binary + fully unbalanced                         0.4377                  0.4105                  0.4237
Multiclass                                        0.4272                  0.4206                  0.4241


Macro F1 (binary setups only; not distorted by class imbalance)

Setup                                        No Transfer          Naive Transfer       Weighted Transfer
-----------------------------------------------------

Note that the sample size is rather low and the variance on these results are high.

In my sample result I found that the Macro F1 results reveal that without the upsampling/downsampling balancing that the paper employs, the the results suffer greatly generally. When considering the paper's novel methodology of weighted knowledge transfer, it did give a meaningful boost to performance in the sample of +0.0958 over no transfer, but this gain did not hold across other data balancing schemes or in the multiclass result.